# 03 — Autograd and Gradients

## Learning Objectives

By the end of this notebook you will understand:

- Why gradients are needed in deep learning
- `requires_grad`
- Computational graphs
- `grad_fn`
- `.grad`
- `.backward()`
- Gradient accumulation and clearing gradients
- `detach()`
- `torch.no_grad()`
- `torch.inference_mode()`
- Leaf and non-leaf tensors
- Non-scalar outputs and the `gradient` argument
- `retain_graph`
- `create_graph`
- Higher-order derivatives
- `torch.autograd.grad`
- How autograd connects to gradient descent
- A complete one-neuron training example

> **Core idea:** Autograd automatically applies the chain rule through a computational graph so that PyTorch can calculate the derivatives needed for training.

# 1. Why Do We Need Gradients?

Suppose a model has a parameter `w` and produces a loss `L`.

We want to change `w` so that the loss becomes smaller.

The derivative

```text
dL/dw
```

tells us how the loss changes when `w` changes.

A basic gradient-descent update is:

```text
w_new = w_old - learning_rate × dL/dw
```

So the training process is:

```text
Loss
 ↓
Gradient
 ↓
Parameter update
 ↓
New prediction
 ↓
New loss
```

# 2. A Simple Derivative

Consider:

```text
y = x²
```

Mathematically:

```text
dy/dx = 2x
```

At `x = 3`:

```text
dy/dx = 6
```

Let's calculate it with PyTorch.

In [1]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

print("x =", x)
print("y =", y)

y.backward()

print("dy/dx =", x.grad)

x = tensor(3., requires_grad=True)
y = tensor(9., grad_fn=<PowBackward0>)
dy/dx = tensor(6.)


The result is `6`.

Notice the important sequence:

```python
requires_grad=True
```

then:

```python
y.backward()
```

then:

```python
x.grad
```

`requires_grad=True` tells PyTorch to track operations involving `x`.

It does **not** calculate the gradient immediately.

# 3. `requires_grad`

Syntax:

```python
torch.tensor(data, requires_grad=True)
```

When enabled, PyTorch records operations needed to calculate derivatives.

Compare the two cases.

In [2]:
x = torch.tensor(3.0)
y = x ** 2

print("Without requires_grad:")
print("x.requires_grad:", x.requires_grad)
print("y.requires_grad:", y.requires_grad)

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

print("\nWith requires_grad:")
print("x.requires_grad:", x.requires_grad)
print("y.requires_grad:", y.requires_grad)
print("y.grad_fn:", y.grad_fn)

Without requires_grad:
x.requires_grad: False
y.requires_grad: False

With requires_grad:
x.requires_grad: True
y.requires_grad: True
y.grad_fn: <PowBackward0 object at 0x00000164DE3678E0>


# 4. Computational Graph

For:

```python
a = x * 2
b = a + 3
c = b ** 2
```

PyTorch builds a computational graph conceptually like:

```text
x
│
× 2
│
▼
a
│
+ 3
│
▼
b
│
²
│
▼
c
```

During backpropagation, PyTorch travels backward through this graph and applies the chain rule.

In [3]:
x = torch.tensor(3.0, requires_grad=True)

a = x * 2
b = a + 3
c = b ** 2

print("a:", a, "| grad_fn:", a.grad_fn)
print("b:", b, "| grad_fn:", b.grad_fn)
print("c:", c, "| grad_fn:", c.grad_fn)

a: tensor(6., grad_fn=<MulBackward0>) | grad_fn: <MulBackward0 object at 0x00000164DC539330>
b: tensor(9., grad_fn=<AddBackward0>) | grad_fn: <AddBackward0 object at 0x00000164DC539330>
c: tensor(81., grad_fn=<PowBackward0>) | grad_fn: <PowBackward0 object at 0x00000164DC539330>


# 5. What is `grad_fn`?

`grad_fn` describes the autograd operation that created a tensor.

You may see names such as:

```text
MulBackward0
AddBackward0
PowBackward0
```

A tensor created directly by the user is generally a **leaf tensor**, so its `grad_fn` is usually `None`.

A tensor created by a tracked operation is generally a **non-leaf tensor**.

In [4]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 3
z = y ** 2

print("x.is_leaf:", x.is_leaf)
print("y.is_leaf:", y.is_leaf)
print("z.is_leaf:", z.is_leaf)

print("\nx.grad_fn:", x.grad_fn)
print("y.grad_fn:", y.grad_fn)
print("z.grad_fn:", z.grad_fn)

x.is_leaf: True
y.is_leaf: False
z.is_leaf: False

x.grad_fn: None
y.grad_fn: <MulBackward0 object at 0x00000164DE394400>
z.grad_fn: <PowBackward0 object at 0x00000164DE394400>


# 6. The Chain Rule

Suppose:

```text
a = 2x
b = a + 3
y = b²
```

The chain rule says:

```text
dy/dx
=
dy/db × db/da × da/dx
```

At `x=3`:

```text
a = 6
b = 9

dy/db = 18
db/da = 1
da/dx = 2

dy/dx = 18 × 1 × 2 = 36
```

PyTorch performs this chain-rule calculation automatically.

In [33]:
x = torch.tensor(3.0, requires_grad=True)

a = 2 * x
b = a + 3
y = b ** 2

y.backward()

print("y =", y.item())
print("dy/dx =", x.grad.item())

y = 81.0
dy/dx = 36.0


# 7. `.backward()`

For a scalar output:

```python
loss.backward()
```

starts reverse-mode automatic differentiation.

Afterward, gradients are normally available in `.grad` of the relevant leaf tensors.

Example:

```text
y = x³
dy/dx = 3x²
```

In [6]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3

y.backward()

print("y:", y.item())
print("dy/dx:", x.grad.item())

y: 8.0
dy/dx: 12.0


# 8. `.grad`

Before backward:

```python
x.grad
```

is normally `None`.

After backward, it contains the accumulated gradient.

In [7]:
x = torch.tensor(4.0, requires_grad=True)

print("Before backward:", x.grad)

y = x ** 2
y.backward()

print("After backward:", x.grad)

Before backward: None
After backward: tensor(8.)


# 9. Gradients Accumulate

This is extremely important.

Calling `backward()` again does not automatically replace the previous gradient.

It adds to it.

For:

```text
y = x²
z = x³
```

at `x=2`:

```text
dy/dx = 4
dz/dx = 12
```

After both backward passes the accumulated gradient is:

```text
16
```

In [8]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
y.backward()
print("After first backward:", x.grad)

z = x ** 3
z.backward()
print("After second backward:", x.grad)

After first backward: tensor(4.)
After second backward: tensor(16.)


# 10. Clearing Gradients

Because gradients accumulate, training loops normally clear old gradients before calculating new ones.

For a raw tensor:

```python
x.grad.zero_()
```

For a model:

```python
optimizer.zero_grad()
```

The optimizer version is what you will normally use in practical training.

In [9]:
x = torch.tensor(2.0, requires_grad=True)

(x ** 2).backward()
print("Gradient:", x.grad)

x.grad.zero_()
print("After clearing:", x.grad)

(x ** 3).backward()
print("New gradient:", x.grad)

Gradient: tensor(4.)
After clearing: tensor(0.)
New gradient: tensor(12.)


# 11. Gradient Descent Connection

Suppose:

```text
L = w²
```

Then:

```text
dL/dw = 2w
```

Gradient descent is:

```text
w_new = w_old - lr × gradient
```

Let's perform one update manually.

In [10]:
w = torch.tensor(5.0, requires_grad=True)
learning_rate = 0.1

loss = w ** 2
loss.backward()

gradient = w.grad

with torch.no_grad():
    new_w = w - learning_rate * gradient

print("Old w:", w.item())
print("Loss:", loss.item())
print("Gradient:", gradient.item())
print("New w:", new_w.item())

Old w: 5.0
Loss: 25.0
Gradient: 10.0
New w: 4.0


At `w=5`:

```text
gradient = 10

new_w = 5 - 0.1 × 10
      = 4
```

This is the bridge between autograd and optimization:

```text
Autograd → gradient → optimizer/update → new parameters
```

# 12. Multiple Parameters

A linear model is:

```text
y = wx + b
```

Both `w` and `b` are learnable parameters.

For:

```text
y = wx + b
```

we have:

```text
dy/dw = x
dy/db = 1
```

In [11]:
x = torch.tensor(2.0)

w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

y = w * x + b
y.backward()

print("y:", y.item())
print("dy/dw:", w.grad.item())
print("dy/db:", b.grad.item())

y: 7.0
dy/dw: 2.0
dy/db: 1.0


# 13. A Loss Function

Training normally compares a prediction against a target.

For a simple example:

```text
prediction = wx + b

loss = (prediction - target)²
```

The loss tells us how wrong the model is.

The gradients tell us how the parameters should change to reduce that loss.

In [12]:
x = torch.tensor(2.0)
target = torch.tensor(10.0)

w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

prediction = w * x + b
loss = (prediction - target) ** 2

loss.backward()

print("Prediction:", prediction.item())
print("Target:", target.item())
print("Loss:", loss.item())
print("dLoss/dw:", w.grad.item())
print("dLoss/db:", b.grad.item())

Prediction: 7.0
Target: 10.0
Loss: 9.0
dLoss/dw: -12.0
dLoss/db: -6.0


# 14. Non-Scalar Outputs

`backward()` is simplest for a scalar.

If:

```python
y = torch.tensor([1., 4., 9.])
```

then:

```python
y.backward()
```

does not know how the individual outputs should be combined.

One simple solution is to reduce the tensor to a scalar:

```python
y.sum().backward()
```

In [13]:
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2

try:
    y.backward()
except RuntimeError as e:
    print("Expected error:")
    print(e)

Expected error:
grad can be implicitly created only for scalar outputs


In [14]:
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2

y.sum().backward()

print("y:", y)
print("x.grad:", x.grad)

y: tensor([1., 4., 9.], grad_fn=<PowBackward0>)
x.grad: tensor([2., 4., 6.])


For:

```text
y = [x₁², x₂², x₃²]
```

the derivative of the sum is:

```text
[2x₁, 2x₂, 2x₃]
```

At `[1,2,3]`:

```text
[2,4,6]
```

# 15. The `gradient` Argument

For a non-scalar output, you can explicitly provide an upstream gradient:

```python
y.backward(gradient=...)
```

This calculates a vector-Jacobian product.

Think of the supplied tensor as specifying how strongly each output contributes to the backward calculation.

In [ ]:
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2

upstream = torch.tensor([1., 1., 1.])

y.backward(gradient=upstream)
    
print("x.grad:", x.grad)

x.grad: tensor([2., 4., 6.])


Change the upstream gradient to:

```text
[1, 2, 3]
```

and observe the result.

In [16]:
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2

y.backward(gradient=torch.tensor([1., 2., 3.]))

print("x.grad:", x.grad)

x.grad: tensor([ 2.,  8., 18.])


# 16. `retain_grad()`

By default, gradients are normally retained for leaf tensors.

Intermediate tensors do not normally keep their `.grad`.

If you want to inspect an intermediate gradient, call:

```python
tensor.retain_grad()
```

In [17]:
x = torch.tensor(2.0, requires_grad=True)

y = x * 3
y.retain_grad()

z = y ** 2
z.backward()

print("x.grad:", x.grad)
print("y.grad:", y.grad)

x.grad: tensor(36.)
y.grad: tensor(12.)


Without `retain_grad()`, `y.grad` would normally be `None`.

This is especially useful when debugging gradient flow through a network.

# 17. `detach()`

`detach()` creates a tensor that shares the data but is disconnected from the current autograd graph.

```python
z = y.detach()
```

After detaching:

```python
z.requires_grad
```

is normally `False`.

In [36]:
x = torch.tensor(3.0, requires_grad=True)

y = x * 2
z = y.detach()

print("y:", y)
print("z:", z)
print("x.requires_grad:", x.requires_grad)
print("y.requires_grad:", y.requires_grad)
print("z.requires_grad:", z.requires_grad)
print("z.grad_fn:", z.grad_fn)

y: tensor(6., grad_fn=<MulBackward0>)
z: tensor(6.)
x.requires_grad: True
y.requires_grad: True
z.requires_grad: False
z.grad_fn: None


Common reasons to use `detach()`:

- Stop gradient flow.
- Use model outputs for logging.
- Separate a tensor from an existing graph.
- Prepare values for operations that should not affect training.

A classic example is inspecting model predictions without backpropagating through them.

# 18. `torch.no_grad()`

Use:

```python
with torch.no_grad():
    ...
```

when operations do not need autograd tracking.

This is common during inference.

In [19]:
x = torch.tensor(3.0, requires_grad=True)

with torch.no_grad():
    y = x ** 2

print("y:", y)
print("requires_grad:", y.requires_grad)
print("grad_fn:", y.grad_fn)

y: tensor(9.)
requires_grad: False
grad_fn: None


### Why use `no_grad()`?

It can:

- Avoid constructing autograd history.
- Reduce memory use.
- Make evaluation more efficient.

Typical pattern:

```python
model.eval()

with torch.no_grad():
    predictions = model(inputs)
```

# 19. `torch.inference_mode()`

`inference_mode()` is another context designed for inference.

Typical usage:

```python
model.eval()

with torch.inference_mode():
    predictions = model(inputs)
```

It disables autograd-related tracking more aggressively and can improve inference performance in appropriate workloads.

In [20]:
x = torch.tensor(3.0, requires_grad=True)

with torch.inference_mode():
    y = x ** 2

print("y:", y)
print("requires_grad:", y.requires_grad)

y: tensor(9.)
requires_grad: False


# 20. `requires_grad_()`

You can change the gradient-tracking setting in-place:

```python
x.requires_grad_(True)
x.requires_grad_(False)
```

The trailing `_` indicates an in-place modification.

In [21]:
x = torch.tensor([1., 2., 3.])

print("Initial:", x.requires_grad)

x.requires_grad_(True)
print("Enabled:", x.requires_grad)

x.requires_grad_(False)
print("Disabled:", x.requires_grad)

Initial: False
Enabled: True
Disabled: False


# 21. Freezing Parameters

Setting:

```python
parameter.requires_grad_ = False
```

means gradients will not be calculated for that parameter.

This is important in transfer learning:

```text
Pretrained layers
       ↓
     frozen
       ↓
New trainable head
```

We will study this later in the course.

In [22]:
weight = torch.tensor([1., 2., 3.], requires_grad=True)

weight.requires_grad_(False)

print("Requires gradient:", weight.requires_grad)

Requires gradient: False


# 22. `torch.autograd.grad()`

Instead of accumulating a gradient into `.grad`, you can request it directly:

```python
torch.autograd.grad(outputs, inputs)
```

In [23]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

gradient = torch.autograd.grad(y, x)

print("Returned gradient:", gradient)

Returned gradient: (tensor(6.),)


### `backward()` vs `autograd.grad()`

`backward()`:

```python
loss.backward()
```

is commonly used in model training and accumulates gradients in parameters.

`autograd.grad()`:

```python
g = torch.autograd.grad(output, input)
```

returns the requested gradient directly.

`autograd.grad()` is especially useful for advanced differentiation, custom algorithms, and research.

# 23. Higher-Order Derivatives

Autograd can differentiate a gradient.

For:

```text
y = x³
```

we have:

```text
dy/dx = 3x²

d²y/dx² = 6x
```

At `x=2`:

```text
first derivative  = 12
second derivative = 12
```

In [37]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 3

first = torch.autograd.grad(
    y, x, create_graph=True
)[0]

second = torch.autograd.grad(
    first, x
)[0]

print(first)

print("y:", y.item())
print("First derivative:", first.item())
print("Second derivative:", second.item())

tensor(12., grad_fn=<MulBackward0>)
y: 8.0
First derivative: 12.0
Second derivative: 12.0


## Why `create_graph=True`?

The first gradient must itself remain differentiable.

```python
create_graph=True
```

tells PyTorch to construct a graph for the gradient computation.

That enables higher-order derivatives.

# 24. `retain_graph=True`

Normally, after backward, PyTorch frees parts of the graph to save memory.

If you need to backpropagate through the **same graph again**, you may use:

```python
retain_graph=True
```

In [25]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2

y.backward(retain_graph=True)
print("After first backward:", x.grad)

y.backward()
print("After second backward:", x.grad)

After first backward: tensor(4.)
After second backward: tensor(8.)


Do not confuse:

```text
retain_graph=True
```

with:

```text
create_graph=True
```

| Parameter | Purpose |
|---|---|
| `retain_graph=True` | Keep the existing graph for another backward pass |
| `create_graph=True` | Build a graph for the gradient calculation itself |

They solve different problems.

# 25. Manual Gradient Verification

We can numerically approximate a derivative with finite differences.

For:

```text
f(x) = x²
```

we can approximate:

```text
f'(x) ≈ [f(x+h) - f(x-h)] / (2h)
```

This is useful for checking custom gradient implementations.

In [39]:
x_value = 3.0
h = 1e-5

def f(x):
    return x ** 2

numerical = (
    f(x_value + h) - f(x_value - h)
) / (2 * h)

print("Numerical:", numerical)
print("Exact:", 2 * x_value)

Numerical: 6.000000000039306
Exact: 6.0


In [27]:
x = torch.tensor(3.0, requires_grad=True)

y = x ** 2
y.backward()

print("PyTorch autograd:", x.grad.item())

PyTorch autograd: 6.0


The numerical approximation and autograd result should be very close.

PyTorch does not normally use finite differences for neural-network training; it uses automatic differentiation.

# 26. `nn.Parameter`

Neural-network learnable values are commonly represented using:

```python
nn.Parameter
```

They normally have `requires_grad=True`.

In [28]:
import torch.nn as nn

weight = nn.Parameter(torch.tensor(2.0))
bias = nn.Parameter(torch.tensor(1.0))

x = torch.tensor(3.0)

prediction = weight * x + bias
loss = (prediction - 10) ** 2

loss.backward()

print("Weight:", weight.item())
print("Weight gradient:", weight.grad.item())

print("Bias:", bias.item())
print("Bias gradient:", bias.grad.item())

Weight: 2.0
Weight gradient: -18.0
Bias: 1.0
Bias gradient: -6.0


This is the bridge from raw autograd to `nn.Module`.

A neural network is essentially a structured collection of parameters and operations whose gradients are calculated by autograd.

# 27. Complete One-Neuron Training Example

Let's train:

```text
prediction = wx + b
```

using:

```text
loss = (prediction - target)²
```

We will manually perform:

```text
Forward
   ↓
Loss
   ↓
Backward
   ↓
Gradient
   ↓
Parameter update
   ↓
Clear gradients
   ↓
Repeat
```

In [40]:
x = torch.tensor(2.0)
target = torch.tensor(10.0)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

learning_rate = 0.01

for step in range(50):
    prediction = w * x + b
    loss = (prediction - target) ** 2

    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    w.grad.zero_()
    b.grad.zero_()

    print(
        f"Step {step+1:2d} | "
        f"prediction={prediction.item():8.4f} | "
        f"loss={loss.item():8.4f} | "
        f"w={w.item():8.4f} | "
        f"b={b.item():8.4f}"
    )

Step  1 | prediction=  2.0000 | loss= 64.0000 | w=  1.3200 | b=  0.1600
Step  2 | prediction=  2.8000 | loss= 51.8400 | w=  1.6080 | b=  0.3040
Step  3 | prediction=  3.5200 | loss= 41.9904 | w=  1.8672 | b=  0.4336
Step  4 | prediction=  4.1680 | loss= 34.0122 | w=  2.1005 | b=  0.5502
Step  5 | prediction=  4.7512 | loss= 27.5499 | w=  2.3104 | b=  0.6552
Step  6 | prediction=  5.2761 | loss= 22.3154 | w=  2.4994 | b=  0.7497
Step  7 | prediction=  5.7485 | loss= 18.0755 | w=  2.6694 | b=  0.8347
Step  8 | prediction=  6.1736 | loss= 14.6411 | w=  2.8225 | b=  0.9113
Step  9 | prediction=  6.5563 | loss= 11.8593 | w=  2.9603 | b=  0.9801
Step 10 | prediction=  6.9006 | loss=  9.6061 | w=  3.0842 | b=  1.0421
Step 11 | prediction=  7.2106 | loss=  7.7809 | w=  3.1958 | b=  1.0979
Step 12 | prediction=  7.4895 | loss=  6.3025 | w=  3.2962 | b=  1.1481
Step 13 | prediction=  7.7406 | loss=  5.1051 | w=  3.3866 | b=  1.1933
Step 14 | prediction=  7.9665 | loss=  4.1351 | w=  3.4679 | b= 

The loss should generally decrease.

This tiny example already contains the essential mechanics of neural-network training.

In real PyTorch:

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

will replace our manual parameter update and gradient clearing.

# 28. Complete Linear Regression with Autograd

Dataset:

```text
x = [1,2,3,4,5]
y = [3,5,7,9,11]
```

The hidden relationship is:

```text
y = 2x + 1
```

We will learn `w` and `b` using only basic autograd.

In [30]:
x = torch.tensor([1., 2., 3., 4., 5.])
target = torch.tensor([3., 5., 7., 9., 11.])

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

learning_rate = 0.01

for step in range(100):
    prediction = w * x + b
    loss = ((prediction - target) ** 2).mean()

    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    w.grad.zero_()
    b.grad.zero_()

print("Learned weight:", w.item())
print("Learned bias:", b.item())
print("Final loss:", loss.item())

Learned weight: 2.081777811050415
Learned bias: 0.7047561407089233
Final loss: 0.01597408391535282


# 29. Why `no_grad()` Is Used During the Update

We want:

```text
w ← w - learning_rate × gradient
```

to change the parameter.

We **do not** want this update operation to become part of the next computational graph.

Therefore:

```python
with torch.no_grad():
    ...
```

is used for manual parameter updates.

This keeps training graphs clean and prevents the optimizer step itself from becoming part of the differentiation history.

# 30. Complete Training Picture

The full relationship is:

```text
                 FORWARD PASS
                      │
                      ▼
Input ───────► Model Parameters ───────► Prediction
                                             │
                                             ▼
                                           Loss
                                             │
                                             ▼
                                      backward()
                                             │
                                             ▼
                                      Computational
                                          Graph
                                             │
                                             ▼
                                        Gradients
                                             │
                                             ▼
                                      Optimizer Step
                                             │
                                             ▼
                                      New Parameters
                                             │
                                             └──── repeat
```

Autograd handles the differentiation part.

The optimizer handles the parameter-update strategy.

# 31. Important Autograd Parameters and APIs

| API / Parameter | Purpose |
|---|---|
| `requires_grad=True` | Track operations for gradients |
| `.grad` | Access accumulated gradient |
| `.backward()` | Run reverse-mode differentiation |
| `.grad_fn` | Show the operation that created a tensor |
| `.is_leaf` | Check whether a tensor is a leaf |
| `.retain_grad()` | Keep gradient for a non-leaf tensor |
| `.detach()` | Disconnect tensor from its current graph |
| `torch.no_grad()` | Disable gradient tracking in a context |
| `torch.inference_mode()` | Efficient inference context |
| `requires_grad_()` | Change gradient tracking in-place |
| `torch.autograd.grad()` | Directly return selected gradients |
| `gradient=` | Supply upstream gradient for non-scalar output |
| `retain_graph=True` | Preserve graph for another backward |
| `create_graph=True` | Make gradients differentiable |

# 32. Common Mistakes

### ❌ Forgetting `requires_grad=True`

PyTorch will not track the required computation.

### ❌ Forgetting to clear gradients

Gradients accumulate.

Use:

```python
optimizer.zero_grad()
```

during normal model training.

### ❌ Calling `backward()` on a non-scalar

Reduce it or supply the `gradient` argument.

### ❌ Updating parameters with gradient tracking enabled

Use:

```python
with torch.no_grad():
    ...
```

for manual updates.

### ❌ Expecting every tensor to have `.grad`

Intermediate tensors normally do not retain gradients unless `retain_grad()` is called.

### ❌ Confusing `detach()` and `no_grad()`

`detach()` disconnects a tensor.

`no_grad()` disables tracking for operations inside its context.

### ❌ Confusing `retain_graph` and `create_graph`

They solve different problems.

# 34. Final Challenge 🚀

Build a tiny linear-regression training loop using only:

- `torch.tensor`
- `requires_grad`
- tensor arithmetic
- `backward()`
- `torch.no_grad()`

Do **not** use `nn.Linear` or an optimizer yet.

Dataset:

```text
x = [1, 2, 3, 4, 5]
y = [3, 5, 7, 9, 11]
```

Start:

```text
w = 0
b = 0
```

Use:

```text
prediction = wx + b
loss = mean((prediction - y)²)
```

Train for 100 steps.

Your final parameters should be approximately:

```text
w ≈ 2
b ≈ 1
```

Required sequence:

```text
Forward
 ↓
Loss
 ↓
backward()
 ↓
Gradients
 ↓
Parameter update
 ↓
Clear gradients
 ↓
Repeat
```

In [31]:
# Try writing the solution yourself first.

x = torch.tensor([1., 2., 3., 4., 5.])
target = torch.tensor([3., 5., 7., 9., 11.])

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

lr = 0.01

for step in range(100):
    prediction = w * x + b
    loss = ((prediction - target) ** 2).mean()

    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    w.grad.zero_()
    b.grad.zero_()

print("w =", w.item())
print("b =", b.item())
print("loss =", loss.item())

w = 2.081777811050415
b = 0.7047561407089233
loss = 0.01597408391535282


# 35. Knowledge Check

Before continuing, you should be able to explain:

1. What is a derivative?
2. Why does deep learning need gradients?
3. What does `requires_grad=True` do?
4. What is a computational graph?
5. What is `grad_fn`?
6. What is a leaf tensor?
7. What does `.backward()` do?
8. Where are gradients stored?
9. Why do gradients accumulate?
10. Why do we clear gradients?
11. Why does a non-scalar output need a reduction or explicit gradient?
12. What does `detach()` do?
13. What is `torch.no_grad()`?
14. What is `torch.inference_mode()`?
15. What does `retain_grad()` do?
16. Difference between `retain_graph=True` and `create_graph=True`?
17. How does the chain rule produce backpropagation?
18. Why do manual parameter updates use `no_grad()`?
19. How does autograd connect to gradient descent?
20. What is the complete training sequence?